# Bloque 6 - Big Data / Spark (Tema 7)
Cooperativa de Rosas (Llagamarca, Baños del Inca, Cajamarca)

Ejecutar en Google Colab (capa gratuita) o Databricks Community Edition. Se usa `cosechas_limpias.csv` (salida del ETL del Bloque 4), ampliado sintéticamente para simular más volumen.

**Antes de ejecutar:** subir `cosechas_limpias.csv` al panel de archivos de Colab (carpeta `/content`).

In [25]:
# Instalar PySpark en Colab (ejecutar esta celda primero; ~1 minuto)
!pip install pyspark -q

In [26]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder.appName("CooperativaRosas").getOrCreate()

## Ampliación sintética del dataset (para simular mayor volumen)
Se replica el dataset limpio 500 veces, **idéntico y sin variaciones** (223 × 500 = 111,500 filas), solo para poder comparar tiempos con más volumen. Como las copias son exactas, los totales de las consultas salen multiplicados por 500 y los promedios no cambian: el ejercicio mide tiempo de procesamiento, no descubre información nueva. En producción real se usaría el histórico acumulado de varios años.

In [27]:
import pandas as pd

base = pd.read_csv("cosechas_limpias.csv")
replicas = [base.copy() for _ in range(500)]  # ~100k filas simuladas
dataset_ampliado = pd.concat(replicas, ignore_index=True)
dataset_ampliado.to_csv("cosechas_ampliado.csv", index=False)
print(len(dataset_ampliado), "filas simuladas")

111500 filas simuladas


In [29]:
df_cosechas = spark.read.csv("cosechas_ampliado.csv", header=True, inferSchema=True)
df_cosechas.printSchema()

root
 |-- id_cosecha: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- vivero: string (nullable = true)
 |-- color: string (nullable = true)
 |-- tamano_cm: integer (nullable = true)
 |-- cantidad_paquetes: integer (nullable = true)



## Método de medición (igual para Spark y SQL Server)
Todos los tiempos se miden **sin imprimir resultados** (en Spark, forzando la ejecución con `.collect()`; en SQL Server, con *Discard results after execution*). Antes de medir se hace un calentamiento: Spark lee el CSV y lo deja en memoria (`.cache()` + `.count()`), y en SQL Server se ejecuta una vez cada consulta para llenar la caché. Cada consulta se ejecuta **3 veces** y se reporta la **mediana**. Los resultados se muestran aparte, fuera de la medición.

In [30]:
import statistics

# Calentamiento: leer y analizar el CSV una sola vez, fuera de la medición
df_cosechas.cache()
print(df_cosechas.count(), "filas en memoria")

def medir(nombre, consulta, repeticiones=3):
    """Ejecuta la consulta completa (sin imprimir) y devuelve la mediana en segundos."""
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        consulta().collect()
        tiempos.append(time.perf_counter() - t0)
    mediana = statistics.median(tiempos)
    print(f"{nombre}: {[round(t, 4) for t in tiempos]} s -> mediana {mediana:.4f} s")
    return mediana

111500 filas en memoria


## Uso de DataFrame API y SparkSQL (2 objetos Spark)

In [31]:
# Consulta 1 - SparkSQL: SUM por color y tamaño
df_cosechas.createOrReplaceTempView("cosechas")
q1 = lambda: spark.sql("""
    SELECT color, tamano_cm, SUM(cantidad_paquetes) AS total_paquetes
    FROM cosechas
    GROUP BY color, tamano_cm
    ORDER BY total_paquetes DESC
""")
t_spark_q1 = medir("Spark (SparkSQL) GROUP BY color, tamano_cm", q1)

Spark (SparkSQL) GROUP BY color, tamano_cm: [5.0875, 0.877, 0.9498] s -> mediana 0.9498 s


In [32]:
# Consulta 2 - DataFrame API: promedio por color
q2 = lambda: df_cosechas.groupBy("color").avg("cantidad_paquetes")
t_spark_q2 = medir("Spark (DataFrame API) AVG por color", q2)

Spark (DataFrame API) AVG por color: [0.854, 0.4666, 0.4115] s -> mediana 0.4666 s


In [33]:
# Resultados (fuera de la medición de tiempos)
q1().show()
q2().show()

+--------+---------+--------------+
|   color|tamano_cm|total_paquetes|
+--------+---------+--------------+
|Amarillo|       50|        218500|
|  Rosado|       60|        213500|
|    Rojo|       80|        183500|
|  Blanco|       50|        182000|
|Amarillo|       60|        180000|
|  Blanco|       80|        174000|
|    Rojo|       60|        143000|
|Amarillo|       80|        133000|
|  Rosado|       50|        127000|
|  Blanco|       60|        106000|
|  Rosado|       80|         99000|
|    Rojo|       50|         87000|
+--------+---------+--------------+

+--------+----------------------+
|   color|avg(cantidad_paquetes)|
+--------+----------------------+
|  Blanco|    15.931034482758621|
|Amarillo|    17.716666666666665|
|    Rojo|    15.036363636363637|
|  Rosado|                 17.58|
+--------+----------------------+



## Comparación SQL Server vs Spark (mismas 111,500 filas)

Para que la comparación sea válida, SQL Server ejecuta **las mismas 2 consultas sobre las mismas 111,500 filas** (las 223 cosechas limpias × 500, igual que este notebook) y con **el mismo método de medición**. Los tiempos de Spark son las medianas impresas por las dos celdas de consulta de arriba.

**Cómo medir en SQL Server:** ejecutar en SSMS el archivo `06_bigdata/06_sqlserver_tiempos.sql` (usa la base `DW_CooperativaRosas`, no necesita cargar ningún CSV). Las instrucciones de medición están en los comentarios del archivo.

| Consulta | Volumen | SQL Server (mediana de 3) | Spark (mediana de 3) |
|---|---|---|---|
| 1. `GROUP BY color, tamano_cm` (SUM) | 111,500 filas | **0,057 s** (57 ms) | **0,950 s** (950 ms, SparkSQL) |
| 2. `AVG` por color | 111,500 filas | **0,121 s** (121 ms) | **0,467 s** (467 ms, DataFrame API) |

> **Lectura justa de los tiempos:** con el calentamiento, ninguna medición incluye leer el CSV ni arrancar los ejecutores, y ninguna incluye imprimir resultados, así que SparkSQL y DataFrame API se pueden comparar entre sí y con SQL Server.

## Análisis de eficiencia y escalabilidad

1. **Con el volumen real, el motor adecuado es SQL Server.** La cooperativa tiene 223 cosechas limpias en 10 semanas y 30 ventas; cualquier consulta sobre eso termina de inmediato en un motor relacional, y Spark agrega un costo fijo (iniciar la sesión, planificar y repartir tareas) que no depende del tamaño de los datos.
2. **El ejercicio de 111,500 filas mide tiempo de procesamiento, no el negocio real.** Es la misma tabla repetida 500 veces, así que los totales salen multiplicados por 500 y los promedios no cambian. Sirve para comparar motores con más volumen, no para descubrir información nueva.
3. **Dónde sí se justificaría Spark:** historiales de varios años o de muchos viveros, archivos que ya no caben en la memoria de un solo equipo, o datos semiestructurados que se procesan en varios nodos. Con este dataset no se determinó a partir de qué volumen la diferencia importa; para saberlo habría que repetir la medición con réplicas mayores (por ejemplo ×5,000 y ×50,000) y comparar cómo crece el tiempo de cada motor.
4. **Resultado medido (ejecutado en Google Colab el 02-10-2026; evidencia en `08_evidencias/06_bigdata/`):** SQL Server fue más rápido en las dos consultas: 0,057 s frente a 0,950 s en la consulta 1 (unas 17 veces) y 0,121 s frente a 0,467 s en la consulta 2 (unas 4 veces), siempre con la mediana de 3 ejecuciones sobre las mismas 111,500 filas. Esto es coherente con el punto 1: con un volumen pequeño, Spark paga un costo fijo de planificación y reparto de tareas que SQL Server no paga. La decisión de la cooperativa no cambia: SQL Server es el motor del día a día y Spark solo se justificaría cuando el volumen crezca mucho más (punto 3).